# Feature Evidence Visual Review Dashboard

Ranking 결과가 진짜 좋은지 확인하기 위한 시각화 노트북입니다.

Top feature별로 아래를 확인합니다.

- feature 값과 y 값의 scatter
- Good/Bad feature 분포
- 설비별 feature median 및 Bad rate
- Good/Bad별 feature presence/missingness

실제 데이터에 적용할 때는 `DATA_DIR`, `RESULT_PATH`, `ORDER_FILE_PATTERN`, `Y_COL`만 바꾸면 됩니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Wide toyset default paths
DATA_DIR = PROJECT_ROOT / "data" / "toy_semiconductor_wide"
RESULT_PATH = PROJECT_ROOT / "outputs" / "wide_toy_semiconductor_booster" / "combined_feature_evidence.csv"
ORDER_FILE_PATTERN = "wide_order_{order_id:03d}.csv"
FULL_POOL_FILE_PATTERN = "wide_order_{order_id:03d}_full_pool.csv"

# Review settings
ORDER_ID = 1
TOP_N = 5
Y_COL = "eds_bin_a_wf_mean"
LABEL_COL = "target_bad_a"
FACET_COL = "equipment_name"

print("project:", PROJECT_ROOT)
print("data:", DATA_DIR)
print("result:", RESULT_PATH)

## 1. Load Ranking Result

`combined_feature_evidence.csv`가 없다면 먼저 `feature_booster_wide_toyset.ipynb` 또는 아래 명령을 실행하세요.

```powershell
python examples\run_feature_booster_on_wide_toyset.py
```

In [ ]:
import pandas as pd
from feature_booster.visual_review import load_evidence_report, select_top_features

evidence = load_evidence_report(RESULT_PATH)
top = select_top_features(evidence, order_id=ORDER_ID, top_n=TOP_N)
top[["order_id", "final_rank", "feature_name", "final_score", "presence_type", "direction", "evidence_reason"]]

## 2. Quick Diagnostic Table

그림을 보기 전에 feature-y 상관, Good/Bad presence, 설비 concentration hint를 표로 확인합니다.

In [ ]:
from feature_booster.visual_review import make_review_decision_table

decision_table = make_review_decision_table(
    evidence=evidence,
    data_dir=DATA_DIR,
    order_id=ORDER_ID,
    y_col=Y_COL,
    top_n=TOP_N,
    file_pattern=FULL_POOL_FILE_PATTERN,
    label_col=LABEL_COL,
    facet_col=FACET_COL,
    role_col="booster_sample_role",
)
decision_table

## 3. Plot Top Features

각 feature별로 `feature vs y`, `Good/Bad 분포`, `설비별 확인`, `presence`를 봅니다.

In [ ]:
from feature_booster.visual_review import plot_order_top_features

figures = plot_order_top_features(
    evidence=evidence,
    data_dir=DATA_DIR,
    order_id=ORDER_ID,
    y_col=Y_COL,
    top_n=TOP_N,
    file_pattern=FULL_POOL_FILE_PATTERN,
    label_col=LABEL_COL,
    facet_col=FACET_COL,
    role_col="booster_sample_role",
)
len(figures)

## 4. Check Full-Pool Process Windows

Top feature가 전체 wafer pool에서 특정 run 구간에만 튀는지 확인합니다. 회색은 booster에서 쓰지 않은 ignored wafer, 파랑은 Good, 빨강은 Bad입니다.

In [ ]:
from feature_booster.visual_review import plot_order_top_feature_process_windows

window_figures = plot_order_top_feature_process_windows(
    evidence=evidence,
    data_dir=DATA_DIR,
    order_id=ORDER_ID,
    y_col=Y_COL,
    top_n=min(TOP_N, 3),
    file_pattern=FULL_POOL_FILE_PATTERN,
    label_col=LABEL_COL,
    time_col="process_run_seq",
    role_col="booster_sample_role",
)
len(window_figures)

## 5. Try Other Checks

`Y_COL`을 바꿔서 같은 feature가 다른 EDS 지표와도 같이 움직이는지 확인하세요.

예시:

```python
Y_COL = "eds_bin_no_wf_mean"
FACET_COL = "chamber_id"
ORDER_ID = 2
TOP_N = 10
```

실제 데이터에서는 `FACET_COL`을 `equipment_name`, `chamber_id`, `lot_id`, `product_id`, `process_step`으로 바꿔가며 confounding을 확인하는 걸 추천합니다.